# spearman

In [ ]:
import numpy as np
from scipy.stats import spearmanr

def spearman_ci(s1, s2, n_boot=2000, ci=0.95, seed=0):
    s1 = np.asarray(s1).ravel()
    s2 = np.asarray(s2).ravel()
    assert s1.shape == s2.shape
    n = s1.shape[0]

    rho = spearmanr(s1, s2).correlation

    rng = np.random.default_rng(seed)
    rhos = np.empty(n_boot, dtype=float)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)  # resample examples with replacement
        rhos[b] = spearmanr(s1[idx], s2[idx]).correlation

    alpha = (1 - ci) / 2
    lo = np.quantile(rhos, alpha)
    hi = np.quantile(rhos, 1 - alpha)
    return float(rho), float(lo), float(hi)




In [14]:
# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt
# import json
# import os
# from pathlib import Path
# import torch
# from scipy.stats import spearmanr
# from tqdm import tqdm




# original_diffmean_path = "../projection_pca_64"
# original_logistic_path = "../linear_model/pca_64/projection_pca_logistic"
# original_hinge_path = "../linear_model/pca_64/projection_pca_hinge"

# file_paths = {
#     "diffmean": original_diffmean_path,
#     "logistic": original_logistic_path,
#     "hinge": original_hinge_path,
# }

# files = {
#     "diffmean": os.listdir(original_diffmean_path),
#     "logistic": os.listdir(original_logistic_path),
#     "hinge": os.listdir(original_hinge_path),
# }


# projected_path = "projection_within_real"

# projected_files = os.listdir(projected_path)
# spearman_corr_real = []



# for metric in ['diffmean', 'hinge', 'logistic']:
#     primary_files = files[metric]


#     for fname in tqdm(primary_files):
#         file = torch.load(os.path.join(file_paths[metric], fname))
        
#         benchmark1 = file['benchmark']
#         column1 = file['column']
#         model = file['model_name'].split('/')[-1]
        
#         dataset = pd.read_csv(f"../{benchmark1}.csv")
    
#         indices = dataset[dataset['dataset']=='test']['id'].tolist()
#         indices = np.array(indices)
#         del dataset
        
#         for pfile in projected_files:
#             projected_files_file = torch.load(os.path.join(projected_path, pfile))
#             if benchmark1 != projected_files_file['benchmark']:
#                 continue
#             if column1 != projected_files_file['column']:
#                 continue
#             if model != projected_files_file['model_name'].split('/')[-1]:
#                 continue
#             if metric == projected_files_file['metric']:
#                 continue


            
#             benchmark2 = projected_files_file['primary_benchmark']
#             column2 = projected_files_file['primary_column']
            
            
#             projection1 = file['projection']
#             projection2 = projected_files_file['projection']
            
#             N, num_layers = projection1.shape
            
 


#             spearman_corr =  np.zeros((num_layers,))
#             spearman_corr_ci = np.zeros((num_layers, 2))
    
#             for layer_index in range(num_layers):

#                 projection1_test_layer = projection1[:, layer_index][indices].numpy()
#                 projection2_test_layer = projection2[:, layer_index][indices].numpy()

#                 spearman_corr[layer_index], spearman_corr_ci[layer_index][0], spearman_corr_ci[layer_index][1] = spearman_ci(projection1_test_layer, projection2_test_layer)

#             result = {
#                 'benchmark1': benchmark1,
#                 'column1': column1,
#                 'benchmark2': benchmark2,
#                 'column2': column2,
#                 'model': model,
#                 "metric" : metric,
#                 'spearman_corr': spearman_corr.tolist(),
#                 'spearman_ci': spearman_corr_ci.tolist(),
#             }

#             spearman_corr_real.append(result)
            
# with open('spearman_corr_cross_real.jsonl', 'w') as f:
#     for item in spearman_corr_real:
#         f.write(json.dumps(item) + "\n")


# jaccard

In [15]:
import numpy as np

def top_bottom_jaccard(scores_a, scores_b, frac=0.10):
    """
    scores_a, scores_b: 1D lists/arrays of length N (one score per row)
    frac: fraction in (0, 1], e.g. 0.10 for top/bottom 10%

    Returns: dict with top/bottom sets and their Jaccard indices.
    """
    a = np.asarray(scores_a).ravel()
    b = np.asarray(scores_b).ravel()
    assert a.shape == b.shape, "scores must have same length"
    n = a.shape[0]
    k = max(1, int(round(frac * n)))

    # indices of top-k
    top_a = set(np.argpartition(a, -k)[-k:].tolist())
    top_b = set(np.argpartition(b, -k)[-k:].tolist())

    # indices of bottom-k
    bot_a = set(np.argpartition(a,  k)[:k].tolist())
    bot_b = set(np.argpartition(b,  k)[:k].tolist())

    def jaccard(A, B):
        union = A | B
        return len(A & B) / len(union) if union else 0.0

    return jaccard(top_a, top_b), jaccard(bot_a, bot_b)

In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import os
from pathlib import Path
import torch
from scipy.stats import spearmanr
from tqdm import tqdm




original_diffmean_path = "../projection_pca_64"
original_logistic_path = "../linear_model/pca_64/projection_pca_logistic"
original_hinge_path = "../linear_model/pca_64/projection_pca_hinge"

file_paths = {
    "diffmean": original_diffmean_path,
    "logistic": original_logistic_path,
    "hinge": original_hinge_path,
}

files = {
    "diffmean": os.listdir(original_diffmean_path),
    "logistic": os.listdir(original_logistic_path),
    "hinge": os.listdir(original_hinge_path),
}


projected_path = "projection_within_real"

projected_files = os.listdir(projected_path)
jaccard_index_real = []



for metric in ['diffmean', 'hinge', 'logistic']:
    primary_files = files[metric]


    for fname in tqdm(primary_files):
        file = torch.load(os.path.join(file_paths[metric], fname))

        benchmark1 = file['benchmark']
        column1 = file['column']
        model = file['model_name'].split('/')[-1]

        dataset = pd.read_csv(f"../{benchmark1}.csv")

        indices = dataset[dataset['dataset']=='test']['id'].tolist()
        indices = np.array(indices)
        del dataset

        for pfile in projected_files:
            projected_files_file = torch.load(os.path.join(projected_path, pfile))
            if benchmark1 != projected_files_file['benchmark']:
                continue
            if column1 != projected_files_file['column']:
                continue
            if model != projected_files_file['model_name'].split('/')[-1]:
                continue
            if metric == projected_files_file['metric']:
                continue


            benchmark2 = projected_files_file['primary_benchmark']
            column2 = projected_files_file['primary_column']


            projection1 = file['projection']
            projection2 = projected_files_file['projection']

            N, num_layers = projection1.shape


            jaccard_index_top =  np.zeros((num_layers,))
            jaccard_index_bottom =  np.zeros((num_layers,))
            

            for layer_index in range(num_layers):

                projection1_test_layer = projection1[:, layer_index][indices].numpy()
                projection2_test_layer = projection2[:, layer_index][indices].numpy()

                jaccard_index_top[layer_index], jaccard_index_bottom[layer_index] = top_bottom_jaccard(projection1_test_layer, projection2_test_layer)

            result = {
                'benchmark1': benchmark1,
                'column1': column1,
                'benchmark2': benchmark2,
                'column2': column2,
                'model': model,
                "metric" : metric,
                'jaccard_index_top': jaccard_index_top.tolist(),
                'jaccard_index_bottom': jaccard_index_bottom.tolist(),
            }

            jaccard_index_real.append(result)

with open('jaccard_index_cross_real.jsonl', 'w') as f:
    for item in jaccard_index_real:
        f.write(json.dumps(item) + "\n")

100%|██████████| 25/25 [00:03<00:00,  6.45it/s]


In [19]:
jaccard = []

with open('jaccard_index_cross_real.jsonl', 'r') as f:
    for line in f:
        jaccard.append(json.loads(line))

len(jaccard)

604